In [ ]:
!pip install Sastrawi nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 8.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
import string
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Data Understanding.csv to Data Understanding.csv


In [ ]:
df = pd.read_csv('Data Understanding.csv', sep=';', encoding='utf-8-sig')
df.head()

,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username,full_text_indonesia
0,"1,88502E+18",Thu Jan 30 19:54:41 +0000 2025,1.0,@elvis_analyst As of January 30 2025 the total...,"1,89E+33",NaN,elvis_analyst,en,NaN,0.0,0.0,0.0,https://x.com/undefined/status/188505404322921...,4872743543.0,NaN,"@elvis_analyst Pada tanggal 30 Januari 2025, t..."
1,"1,88505E+18",Thu Jan 30 19:47:22 +0000 2025,0.0,Toncoinは売り圧力に直面していますが、潜在的な反転シグナルが現れています。 Explo...,"1,89E+34",NaN,NaN,ja,NaN,0.0,0.0,0.0,https://x.com/undefined/status/188505220523926...,"1,61E+34",NaN,"Toncoin menghadapi tekanan jual, tetapi potens..."
2,"1,88505E+18",Thu Jan 30 19:37:59 +0000 2025,1.0,Toncoin'de (TON) dikkatini çekecek sinyaller v...,"1,89E+34",NaN,NaN,tr,NaN,0.0,0.0,0.0,https://x.com/undefined/status/188504984283086...,"1,70E+33",NaN,Toncoin (TON) memiliki sinyal yang akan menari...
3,"1,88505E+18",Thu Jan 30 19:36:49 +0000 2025,0.0,gmonad! I'm reserving both toncoin.nad and ton...,"1,89E+34",NaN,NaN,en,NaN,0.0,0.0,0.0,https://x.com/undefined/status/188504954958191...,"1,72E+34",NaN,bagus sekali! Saya memesan toncoin.nad dan ton...
4,"1,88504E+18",Thu Jan 30 19:15:48 +0000 2025,0.0,gmonad! I'm reserving both toncoin.mon and ton...,"1,89E+34",NaN,NaN,en,NaN,0.0,0.0,0.0,https://x.com/undefined/status/188504425764116...,"1,52E+34",NaN,bagus sekali! Saya memesan toncoin.mon dan ton...


In [ ]:
df = df[['full_text_indonesia']].copy()
df = df.dropna()
df.head()

,full_text_indonesia
0,"@elvis_analyst Pada tanggal 30 Januari 2025, t..."
1,"Toncoin menghadapi tekanan jual, tetapi potens..."
2,Toncoin (TON) memiliki sinyal yang akan menari...
3,bagus sekali! Saya memesan toncoin.nad dan ton...
4,bagus sekali! Saya memesan toncoin.mon dan ton...


In [ ]:
def case_folding(text):
    return text.lower()

df['case_folding'] = df['full_text_indonesia'].astype(str).apply(case_folding)
df[['full_text_indonesia', 'case_folding']].head()

,full_text_indonesia,case_folding
0,"@elvis_analyst Pada tanggal 30 Januari 2025, t...","@elvis_analyst pada tanggal 30 januari 2025, t..."
1,"Toncoin menghadapi tekanan jual, tetapi potens...","toncoin menghadapi tekanan jual, tetapi potens..."
2,Toncoin (TON) memiliki sinyal yang akan menari...,toncoin (ton) memiliki sinyal yang akan menari...
3,bagus sekali! Saya memesan toncoin.nad dan ton...,bagus sekali! saya memesan toncoin.nad dan ton...
4,bagus sekali! Saya memesan toncoin.mon dan ton...,bagus sekali! saya memesan toncoin.mon dan ton...


In [ ]:
def cleaning_data(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)      # hapus URL
    text = re.sub(r'@\w+', '', text)                         # hapus mention
    text = re.sub(r'#\w+', '', text)                         # hapus hashtag
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)              # hapus tanda baca/simbol
    text = re.sub(r'\s+', ' ', text).strip()                 # hapus spasi berlebih
    return text

df['cleaning'] = df['case_folding'].apply(cleaning_data)
df[['case_folding', 'cleaning']].head()

,case_folding,cleaning
0,"@elvis_analyst pada tanggal 30 januari 2025, t...",pada tanggal 30 januari 2025 total pasokan ton...
1,"toncoin menghadapi tekanan jual, tetapi potens...",toncoin menghadapi tekanan jual tetapi potensi...
2,toncoin (ton) memiliki sinyal yang akan menari...,toncoin ton memiliki sinyal yang akan menarik ...
3,bagus sekali! saya memesan toncoin.nad dan ton...,bagus sekali saya memesan toncoin nad dan tonc...
4,bagus sekali! saya memesan toncoin.mon dan ton...,bagus sekali saya memesan toncoin mon dan tonc...


In [ ]:
def remove_number(text):
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['remove_number'] = df['cleaning'].apply(remove_number)
df[['cleaning', 'remove_number']].head()

,cleaning,remove_number
0,pada tanggal 30 januari 2025 total pasokan ton...,pada tanggal januari total pasokan toncoin ton...
1,toncoin menghadapi tekanan jual tetapi potensi...,toncoin menghadapi tekanan jual tetapi potensi...
2,toncoin ton memiliki sinyal yang akan menarik ...,toncoin ton memiliki sinyal yang akan menarik ...
3,bagus sekali saya memesan toncoin nad dan tonc...,bagus sekali saya memesan toncoin nad dan tonc...
4,bagus sekali saya memesan toncoin mon dan tonc...,bagus sekali saya memesan toncoin mon dan tonc...


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
from nltk.tokenize import word_tokenize

def tokenization(text):
    return word_tokenize(text)

df['tokenization'] = df['remove_number'].apply(tokenization)
df[['remove_number', 'tokenization']].head()

,remove_number,tokenization
0,pada tanggal januari total pasokan toncoin ton...,"[pada, tanggal, januari, total, pasokan, tonco..."
1,toncoin menghadapi tekanan jual tetapi potensi...,"[toncoin, menghadapi, tekanan, jual, tetapi, p..."
2,toncoin ton memiliki sinyal yang akan menarik ...,"[toncoin, ton, memiliki, sinyal, yang, akan, m..."
3,bagus sekali saya memesan toncoin nad dan tonc...,"[bagus, sekali, saya, memesan, toncoin, nad, d..."
4,bagus sekali saya memesan toncoin mon dan tonc...,"[bagus, sekali, saya, memesan, toncoin, mon, d..."


In [ ]:
stop_words = set(stopwords.words('indonesian'))

def stopword_removal(tokens):
    return [word for word in tokens if word not in stop_words]

df['stopword_removal'] = df['tokenization'].apply(stopword_removal)
df[['tokenization', 'stopword_removal']].head()

,tokenization,stopword_removal
0,"[pada, tanggal, januari, total, pasokan, tonco...","[tanggal, januari, total, pasokan, toncoin, to..."
1,"[toncoin, menghadapi, tekanan, jual, tetapi, p...","[toncoin, menghadapi, tekanan, jual, potensi, ..."
2,"[toncoin, ton, memiliki, sinyal, yang, akan, m...","[toncoin, ton, memiliki, sinyal, menarik, perh..."
3,"[bagus, sekali, saya, memesan, toncoin, nad, d...","[bagus, memesan, toncoin, nad, toncoin, chog, ..."
4,"[bagus, sekali, saya, memesan, toncoin, mon, d...","[bagus, memesan, toncoin, mon, toncoin, moyaki..."


In [ ]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming(tokens):
    return [stemmer.stem(word) for word in tokens]

df['stemming'] = df['stopword_removal'].apply(stemming)
df[['stopword_removal', 'stemming']].head()

,stopword_removal,stemming
0,"[tanggal, januari, total, pasokan, toncoin, to...","[tanggal, januari, total, pasok, toncoin, ton,..."
1,"[toncoin, menghadapi, tekanan, jual, potensi, ...","[toncoin, hadap, tekan, jual, potensi, sinyal,..."
2,"[toncoin, ton, memiliki, sinyal, menarik, perh...","[toncoin, ton, milik, sinyal, tarik, perhati, ..."
3,"[bagus, memesan, toncoin, nad, toncoin, chog, ...","[bagus, mes, toncoin, nad, toncoin, chog, pesa..."
4,"[bagus, memesan, toncoin, mon, toncoin, moyaki...","[bagus, mes, toncoin, mon, toncoin, moyaki, pe..."


In [ ]:
def join_text(tokens):
    return ' '.join(tokens)

df['hasil_preprocessing'] = df['stemming'].apply(join_text)

In [ ]:
df['tokenization'] = df['tokenization'].apply(lambda x: ' '.join(x))
df['stopword_removal'] = df['stopword_removal'].apply(lambda x: ' '.join(x))
df['stemming'] = df['stemming'].apply(lambda x: ' '.join(x))

df.to_csv('hasil_preprocessing.csv', index=False, sep=';', encoding='utf-8-sig')

In [ ]:
files.download('hasil_preprocessing.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>